In [2]:
import torch
import pandas as pd
from pathlib import Path

from src.prompt_manager import PromptManager
from src.data_manager import DataManager

from src import data_processing
import yaml
from src import paths
import hashlib

from src.analysis.reliability import ReliabilityAnalyzer

In [3]:
with open('../src/configs/config.yaml', 'r') as f:
    full_config = yaml.safe_load(f)

# active_analysis = full_config['active_analysis']
active_analysis = 'mcgill_qa_feedback'
model_vars = full_config['analyses'][active_analysis]['model_vars']
experimental_groups = model_vars['experimental_groups']

In [4]:
raw_df = pd.read_parquet(paths.RAW_DATA_DIR / f'{active_analysis}.parquet')

In [5]:
final_df = data_processing.get_analysis_ready_df(full_config=full_config,
                                                 active_analysis='mcgill_qa_feedback',
                                                 use_cache=False,
                                                 force_refresh=False)

Loading files for analysis mcgill_qa_feedback
🐢 Running full processing pipeline...


KeyboardInterrupt: 

In [ ]:
analyzer = ReliabilityAnalyzer(final_df, group_cols=[
                               'model_name', 'prompt_id', 'dimension_name'], llm_rating_col='mean_rating')
analyzer.compute_reliability_gap(metric='spearman')

,model_name,prompt_id,dimension_name,metric_type,spearman_gap,spearman_hh,spearman_llm_avg
3,Qwen/Qwen3-4B-Instruct-2507,3fc3c2dce92c,Relevance,Spearman,0.000735,0.589539,0.588804
0,Qwen/Qwen3-4B-Instruct-2507,261acd58fe,quality,Spearman,0.006445,0.633026,0.626581
1,Qwen/Qwen3-4B-Instruct-2507,3fc3c2dce92c,Completeness,Spearman,0.024138,0.589539,0.565401
2,Qwen/Qwen3-4B-Instruct-2507,3fc3c2dce92c,Directness,Spearman,0.024380,0.589520,0.565139
11,meta-llama/Llama-3.2-3B-Instruct,3fc3c2dce92c,Relevance,Spearman,0.068403,0.633026,0.564623
4,google/gemma-3-4b-it,261acd58fe,quality,Spearman,0.075549,0.561105,0.485556
5,google/gemma-3-4b-it,3fc3c2dce92c,Completeness,Spearman,0.084609,0.561105,0.476496
9,meta-llama/Llama-3.2-3B-Instruct,3fc3c2dce92c,Completeness,Spearman,0.090214,0.633026,0.542812
7,google/gemma-3-4b-it,3fc3c2dce92c,Relevance,Spearman,0.096643,0.561105,0.464461
10,meta-llama/Llama-3.2-3B-Instruct,3fc3c2dce92c,Directness,Spearman,0.098146,0.633026,0.534881


In [ ]:
analyzer = ReliabilityAnalyzer(
    final_df, group_cols=['model_name', 'prompt_id', 'dimension_name'])
analyzer.analyze_calibration('human_disagreement')

,model_name,prompt_id,dimension_name,calibration_corr
0,Qwen/Qwen3-4B-Instruct-2507,261acd58fe,quality,0.306506
1,Qwen/Qwen3-4B-Instruct-2507,3fc3c2dce92c,Completeness,0.197388
2,Qwen/Qwen3-4B-Instruct-2507,3fc3c2dce92c,Directness,0.175280
3,Qwen/Qwen3-4B-Instruct-2507,3fc3c2dce92c,Relevance,0.210163
4,google/gemma-3-4b-it,261acd58fe,quality,0.042260
5,google/gemma-3-4b-it,3fc3c2dce92c,Completeness,0.042098
6,google/gemma-3-4b-it,3fc3c2dce92c,Directness,0.017291
7,google/gemma-3-4b-it,3fc3c2dce92c,Relevance,-0.018862
8,meta-llama/Llama-3.2-3B-Instruct,261acd58fe,quality,-0.103952
9,meta-llama/Llama-3.2-3B-Instruct,3fc3c2dce92c,Completeness,0.106397


In [6]:
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import scipy.stats as stats

In [ ]:
full_model_df = final_df[['input_id', 'prompt_id', 'model_name', 'mean_human_rating',
                          'mode_rating', 'mean_rating', 'dimension_name', 'normalized_entropy']]

In [20]:
pm = PromptManager(folder=Path(
    "../prompts/PromptSuites/sandbox/MCGILL_QA_FEEDBACK"))
pm.load_all()
prompt_hash_map = {key: item.metadata['description'] for key, item in pm.suites.items()}

PromptManager initialized with folder: ..\prompts\PromptSuites\sandbox\MCGILL_QA_FEEDBACK
Scanning 2 suites from ..\prompts\PromptSuites\sandbox\MCGILL_QA_FEEDBACK...
Loaded 2 PromptSuites


In [ ]:
models = {}

for group_key, group_df in full_model_df.groupby(['model_name', 'prompt_id']):
    if group_df['dimension_name'].unique()[0] == 'quality':
        formula = 'mean_human_rating ~ mean_rating'
        group_df_wide = group_df
    else:
        group_df_wide = group_df.pivot_table(index=['input_id', 'mean_human_rating', 'model_name'],
                                             columns='dimension_name',
                                             values=['normalized_entropy', 'mean_rating']).reset_index()
        group_df_wide.columns = ['_'.join(col).strip(
            '_') if col[1] else col[0] for col in group_df_wide.columns.values]
        formula = "mean_human_rating ~ mean_rating_Relevance + mean_rating_Completeness + mean_rating_Directness"

    model = smf.ols(formula, data=group_df_wide).fit()

    models[str(f"{group_key[0]}_{prompt_hash_map[group_key[1]]}")] = model

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output


def interactive_dataframe_selector(data_dict, description="Select option:"):
    # label -> value pairs; value is the tuple key
    options = [k for k in data_dict.keys()]
    dropdown = widgets.Dropdown(
        options=options,
        description=description,
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )

    def update_table(change):
        clear_output(wait=True)
        display(dropdown)
        display(data_dict[change.new].summary())

    dropdown.observe(update_table, names='value')
    display(dropdown)
    display(data_dict[dropdown.value].summary())

In [ ]:
interactive_dataframe_selector(models)

Dropdown(description='Select option:', layout=Layout(width='400px'), options=('Qwen/Qwen3-4B-Instruct-2507_nai…

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:      mean_human_rating   R-squared:                       0.406
Model:                            OLS   Adj. R-squared:                  0.403
Method:                 Least Squares   F-statistic:                     135.6
Date:                Mon, 02 Feb 2026   Prob (F-statistic):           3.27e-24
Time:                        09:38:34   Log-Likelihood:                -257.66
No. Observations:                 200   AIC:                             519.3
Df Residuals:                     198   BIC:                             525.9
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept       1.1017      0.132      8.378      0.000       0.842       1.361
mean_rating     0.5164      0.044     11.645      0.000       0.429       0.604
==============================================================================
Omnibus:                        1.892   Durbin-Watson:                   1.770
Prob(Omnibus):                  0.388   Jarque-Bera (JB):                1.560
Skew:                          -0.201   Prob(JB):                        0.459
Kurtosis:                       3.159   Cond. No.                         6.82
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""